# xfig_30 — Performance Gain Waterfall

Decomposes AUROC at 240m K=5 Transformer into additive contributions:

1. **Base**: MeanPool @ 30s, K=1 (minimal effort baseline)
2. **+Aggregation**: K=1 → K=5 at 30s (inference-time gain, free)
3. **+Context**: 30s → 240m at K=5 MeanPool (training context gain)
4. **+Architecture**: MeanPool → Transformer at 240m, K=5
5. **=Final**: Transformer @ 240m, K=5

One panel per task. Shows which factor contributes most.

Idea #30 from `docs/NEW_PLOT_IDEAS.md`.

**Data**: specific cells from `analysis.csv`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
EXPLORE_DIR    = NSRR_TOOLS / "results" / "paper_figures" / "explore"
FINAL_OUT      = EXPLORE_DIR / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)
TABLES_DIR     = NSRR_TOOLS / "results" / "tables"

# Add explore utils to path
_nb_dir = EXPLORE_DIR / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.data_explore import (
    set_root, load_analysis, load_analysis_all_k,
    load_heatmap, load_parquets, load_modality_table,
    subject_predictions, subject_correctness_matrix, CONTEXT_TO_MIN, CTX_ORDER,
)
from utils import panels_explore as xp

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

set_root(WORKSPACE_ROOT)
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 7,
})

# ── Constants ──────────────────────────────────────────────────────────────────
MAIN_TASKS = ["sex_binary", "bmi_binary", "age_class",
              "sleep_efficiency_binary", "apnea_binary"]
TASK_LABEL = xp.TASK_LABEL

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND'}")


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TASKS  = MAIN_TASKS
N_COLS = 3
N_ROWS = (len(TASKS) + N_COLS - 1) // N_COLS

# Load analysis.csv with ALL k values (not just k='all')
df_allk = load_analysis_all_k("phase0_v3", split="test")

# Quick sanity check: print K=1 and K=5 for sex_binary/mean_pool/30s
check = df_allk[
    (df_allk.task == "sex_binary") &
    (df_allk.head == "mean_pool") &
    (df_allk.context_length == "30s") &
    (df_allk.k.isin(["1", "5"]))
][["task", "head", "context_length", "k", "mean_prob_auroc"]]
print(check.to_string())

In [ ]:
labels = [chr(97 + i) for i in range(len(TASKS))]
n_last = len(TASKS) % N_COLS or N_COLS
n_full = len(TASKS) // N_COLS

mosaic = []
for row in range(n_full):
    rl = labels[row * N_COLS:(row + 1) * N_COLS]
    mosaic.append([l for l in rl for _ in range(2)])
if n_last < N_COLS:
    pad = N_COLS - n_last
    ll  = labels[n_full * N_COLS:]
    mosaic.append(["."] * pad + [l for l in ll for _ in range(2)] + ["."] * pad)

fig, axd = plt.subplot_mosaic(mosaic, figsize=(7.0, N_ROWS * 2.8))

for i, (lbl, task) in enumerate(zip(labels, TASKS)):
    ax = axd[lbl]
    xp.waterfall_panel(ax, df_allk, task)
    ax.set_title(TASK_LABEL.get(task, task), fontsize=8)
    ax.text(0.02, 0.97, f"({lbl})", transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top")
    # Only leftmost panels of each row get y-label
    if i % N_COLS != 0:
        ax.set_ylabel("")

fig.suptitle("AUROC gain decomposition: aggregation + context + architecture",
             fontsize=8, y=1.01)
fig.tight_layout(h_pad=1.8, w_pad=0.8)
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
fig.savefig(str(FINAL_OUT / 'xfig_30_waterfall.pdf'), bbox_inches='tight')
fig.savefig(str(FINAL_OUT / 'xfig_30_waterfall.png'), dpi=150, bbox_inches='tight')
print('Saved →', FINAL_OUT / 'xfig_30_waterfall.pdf')